In [1]:
# Cell 1 - Configurar rutas
WORKSPACE_ID = "956e9e93-b6e5-451c-b114-91929dbfe82c"
BRONZE_ID = "702e246c-b575-416c-b603-8c63a947e802"
SILVER_ID = "4bb699f7-bd8e-4f14-a3bc-5585f5067393"

# Obtener ID de LH_Gold
lakehouses = mssparkutils.lakehouse.list()
GOLD_ID = None
for lh in lakehouses:
    if lh['displayName'] == 'LH_Gold':
        GOLD_ID = lh['id']
        print(f"LH_Gold ID: {GOLD_ID}")

BRONZE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"
SILVER_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"
GOLD_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

print("Rutas configuradas OK")

StatementMeta(, a100fd7f-4a10-48fb-876a-cb90cc355a57, 3, Finished, Available, Finished, False)

LH_Gold ID: 50572ea4-cca9-46c8-a797-0644b99e54ec
Rutas configuradas OK


In [2]:
# Cell 2 - Crear DimDate (dimensión de tiempo)
from pyspark.sql.functions import col, year, month, dayofmonth, quarter, dayofweek, date_format
from pyspark.sql import Row
import pandas as pd

# Generar tabla de fechas 2010-2026
date_range = pd.date_range(start='2010-01-01', end='2026-12-31', freq='D')
date_df = pd.DataFrame({
    'DateKey': [int(d.strftime('%Y%m%d')) for d in date_range],
    'Date': date_range,
    'Year': [d.year for d in date_range],
    'Quarter': [d.quarter for d in date_range],
    'Month': [d.month for d in date_range],
    'MonthName': [d.strftime('%B') for d in date_range],
    'Day': [d.day for d in date_range],
    'DayOfWeek': [d.dayofweek + 1 for d in date_range],
    'DayName': [d.strftime('%A') for d in date_range],
    'IsWeekend': [1 if d.dayofweek >= 5 else 0 for d in date_range],
    'YearMonth': [d.strftime('%Y-%m') for d in date_range],
    'YearQuarter': [f"{d.year}-Q{d.quarter}" for d in date_range]
})

dim_date = spark.createDataFrame(date_df)
dim_date.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/DimDate")
print(f"OK: DimDate ({dim_date.count():,} filas)")

StatementMeta(, a100fd7f-4a10-48fb-876a-cb90cc355a57, 4, Finished, Available, Finished, False)

OK: DimDate (6,209 filas)


In [5]:
# Cell 3 - Crear dimensiones desde Silver
from pyspark.sql.functions import col

# DimProduct - renombrar columnas antes del join
df_product = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Production_Product")
df_category = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Production_ProductCategory")
df_subcategory = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Production_ProductSubcategory")

df_subcategory_clean = df_subcategory.select(
    col("ProductSubcategoryID"),
    col("Name").alias("SubcategoryName"),
    col("ProductCategoryID")
)

df_category_clean = df_category.select(
    col("ProductCategoryID"),
    col("Name").alias("CategoryName")
)

dim_product = df_product.join(df_subcategory_clean, on="ProductSubcategoryID", how="left") \
    .join(df_category_clean, on="ProductCategoryID", how="left") \
    .select(
        col("ProductID"),
        col("Name").alias("ProductName"),
        col("ProductNumber"),
        col("Color"),
        col("StandardCost"),
        col("ListPrice"),
        col("Size"),
        col("Weight"),
        col("SubcategoryName"),
        col("CategoryName"),
        col("source_year")
    ).distinct()

dim_product.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/DimProduct")
print(f"OK: DimProduct ({dim_product.count():,} filas)")

# DimCustomer
df_customer = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_Customer")
dim_customer = df_customer.select(
    col("CustomerID"),
    col("AccountNumber"),
    col("source_year")
).distinct()

dim_customer.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/DimCustomer")
print(f"OK: DimCustomer ({dim_customer.count():,} filas)")

# DimTerritory
df_territory = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_SalesTerritory")
dim_territory = df_territory.select(
    col("TerritoryID"),
    col("Name").alias("TerritoryName"),
    col("CountryRegionCode"),
    col("Group").alias("TerritoryGroup"),
    col("source_year")
).distinct()

dim_territory.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/DimTerritory")
print(f"OK: DimTerritory ({dim_territory.count():,} filas)")

# DimSalesPerson
df_salesperson = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_SalesPerson")
dim_salesperson = df_salesperson.select(
    col("BusinessEntityID").alias("SalesPersonID"),
    col("TerritoryID"),
    col("SalesQuota"),
    col("Bonus"),
    col("CommissionPct"),
    col("SalesYTD"),
    col("SalesLastYear"),
    col("source_year")
).distinct()

dim_salesperson.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/DimSalesPerson")
print(f"OK: DimSalesPerson ({dim_salesperson.count():,} filas)")

StatementMeta(, a100fd7f-4a10-48fb-876a-cb90cc355a57, 7, Finished, Available, Finished, False)

OK: DimProduct (1,512 filas)
OK: DimCustomer (59,460 filas)
OK: DimTerritory (30 filas)
OK: DimSalesPerson (51 filas)


In [7]:
# Cell 4 - Crear tablas de hechos
from pyspark.sql.functions import col, year, month

# FactSales
df_header = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_SalesOrderHeader")
df_detail = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_SalesOrderDetail")

# Renombrar source_year en detail para evitar ambigüedad
df_detail_clean = df_detail.drop("source_year")

fact_sales = df_detail_clean.join(
    df_header.select(
        col("SalesOrderID"),
        col("OrderDate"),
        col("CustomerID"),
        col("SalesPersonID"),
        col("TerritoryID"),
        col("SubTotal"),
        col("TaxAmt"),
        col("Freight"),
        col("TotalDue"),
        col("Status"),
        col("OnlineOrderFlag"),
        col("source_year")
    ),
    on="SalesOrderID",
    how="inner"
).select(
    col("SalesOrderID"),
    col("SalesOrderDetailID"),
    col("OrderDate"),
    year(col("OrderDate")).alias("OrderYear"),
    month(col("OrderDate")).alias("OrderMonth"),
    col("CustomerID"),
    col("SalesPersonID"),
    col("TerritoryID"),
    col("ProductID"),
    col("OrderQty"),
    col("UnitPrice"),
    col("UnitPriceDiscount"),
    col("LineTotal"),
    col("SubTotal"),
    col("TaxAmt"),
    col("Freight"),
    col("TotalDue"),
    col("Status"),
    col("OnlineOrderFlag"),
    col("source_year")
)

fact_sales.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/FactSales")
print(f"OK: FactSales ({fact_sales.count():,} filas)")

# FactInventory
df_inventory = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Production_ProductInventory")

fact_inventory = df_inventory.select(
    col("ProductID"),
    col("LocationID"),
    col("Shelf"),
    col("Bin"),
    col("Quantity"),
    col("source_year")
)

fact_inventory.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/FactInventory")
print(f"OK: FactInventory ({fact_inventory.count():,} filas)")

# FactWorkOrder
df_workorder = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Production_WorkOrder")

fact_workorder = df_workorder.select(
    col("WorkOrderID"),
    col("ProductID"),
    col("OrderQty"),
    col("StockedQty"),
    col("ScrappedQty"),
    col("StartDate"),
    col("EndDate"),
    col("DueDate"),
    col("ScrapReasonID"),
    col("source_year")
)

fact_workorder.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/Tables/FactWorkOrder")
print(f"OK: FactWorkOrder ({fact_workorder.count():,} filas)")

StatementMeta(, a100fd7f-4a10-48fb-876a-cb90cc355a57, 9, Finished, Available, Finished, False)

OK: FactSales (1,091,853 filas)
OK: FactInventory (3,207 filas)
OK: FactWorkOrder (217,773 filas)


In [8]:
# Cell 5 - Verificación final Gold layer
tablas_gold = mssparkutils.fs.ls(f"{GOLD_PATH}/Tables")
print(f"Total tablas en LH_Gold: {len(tablas_gold)}")
for t in tablas_gold:
    print(f"  - {t.name}")

StatementMeta(, a100fd7f-4a10-48fb-876a-cb90cc355a57, 10, Finished, Available, Finished, False)

Total tablas en LH_Gold: 9
  - DimCustomer
  - DimDate
  - DimProduct
  - DimSalesPerson
  - DimTerritory
  - FactInventory
  - FactSales
  - FactWorkOrder
  - dbo
